In [1]:
import torch

## Building simple attention

In [2]:
#Building simple attention, understanding attention weights
# model can see entire context, assign weights to input texts so far for each query value
# first generate attention weights for a query against all token embeddings or keys
# then context vector for that query is a combination sum of all the input tokens weighted by the attention weights.
# wrd_embeddings = torch.tensor([
#     [0.1,0.3,0.4]
#     ,[0.2,0.5,0.7]
#     ,[0.78,0.323,0.124]
#     ,[0.14,0.63,0.84]
#     ,[0.21,0.35,0.67]
#     ,[0.78,0.323,0.124]
# ])
#quick check against example vectors
wrd_embeddings = torch.tensor(
    [
        [0.43,0.15,0.89]
        ,[.55,.87,.66]
        ,[.57,.85,.64]
        ,[.22,.58,.33]
        ,[.77,.25,.10]
        ,[.05,.80,.55]
    ]
)


In [3]:
print(wrd_embeddings.shape)

torch.Size([6, 3])


In [60]:
# first step compute attention scores for lets say word 2 as the query vector. The attention scores are the unnormalized version of attention weights.
# query = wrd_embeddings[1]
attention_weights = []
for query in wrd_embeddings:
    res = torch.empty(wrd_embeddings.shape[0])
    # print(res)
    for i,emb in enumerate(wrd_embeddings):
        res[i] = torch.dot(query,emb)
    attention_weights.append(res)
print(attention_weights)

[tensor([0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310]), tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865]), tensor([0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605]), tensor([0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565]), tensor([0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935]), tensor([0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450])]


In [71]:
# for attention scores, simplest way to normalize is to divide by the sum
attention_scores_sum = []
for i,t in enumerate(attention_weights):
    # print(i)
    normalized_emb = t/(t.sum())
    attention_scores_sum.append(normalized_emb)

def softmax_naive(x):
    #softmax advantages 1.) differentiable, nicer properties during optimization 2.) tendency to give higher probabilities with larger numbers as an indicator of higher confidence as opposed to normal proportions 3.) can handle negative values and give correct probability assignment
    e = torch.exp(x)
    return e/e.sum()


attention_scores_softmax = []
for i,t in enumerate(attention_weights):
    # print(i)
    # normalized_emb = softmax_naive(t)
    normalized_emb = torch.softmax(t,dim = 0)
    attention_scores_softmax.append(normalized_emb)

print(attention_weights)
print(attention_scores_sum)
print(attention_scores_softmax)


[tensor([0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310]), tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865]), tensor([0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605]), tensor([0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565]), tensor([0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935]), tensor([0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450])]
[tensor([0.2241, 0.2140, 0.2113, 0.1066, 0.1026, 0.1415]), tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656]), tensor([0.1454, 0.2277, 0.2248, 0.1280, 0.1104, 0.1637]), tensor([0.1304, 0.2313, 0.2275, 0.1354, 0.0953, 0.1801]), tensor([0.1436, 0.2219, 0.2245, 0.1090, 0.2088, 0.0921]), tensor([0.1350, 0.2325, 0.2269, 0.1405, 0.0628, 0.2022])]
[tensor([0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452]), tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581]), tensor([0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565]), tensor([0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720]), tensor([0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295]), tensor([0.1

In [ ]:
# generating context vectors
attention_scores_softmax = torch.stack(attention_scores_softmax)
context_vectors = torch.matmul(attention_scores_softmax,wrd_embeddings)

In [ ]:
# quick check
att = attention_scores_softmax[1]
query = wrd_embeddings[1]
context_vec = torch.zeros(query.shape)
for i,emb in enumerate(wrd_embeddings):
    context_vec+=emb*att[i]
print(context_vec)
print(context_vectors[1])

tensor([0.4419, 0.6515, 0.5683])
tensor([0.4419, 0.6515, 0.5683])
tensor([0.4419, 0.6515, 0.5683])


Formalizing above example

In [79]:
attn_scores = wrd_embeddings@wrd_embeddings.T
attn_weights = torch.softmax(attn_scores,dim = 1) #normalized weight should sum to 1 across the row
context_vectors = attn_weights@wrd_embeddings
print(context_vectors)


tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


## Implement self attention mechanism with trainable weights

In [4]:
# Wq, Wk and Wv to generate query, key and value vectors which are intermediate values to generate attention scores and context vectors
wrd_embeddings
x_2 = wrd_embeddings[1]
d_in = wrd_embeddings.shape[1]
d_out = 2

In [84]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in,d_out)) # wrapper that makes weights trainable
W_key = torch.nn.Parameter(torch.rand(d_in,d_out))
W_value = torch.nn.Parameter(torch.rand(d_in,d_out))

In [90]:
query_2 = x_2@W_query #linear transformation from 3 dimensions projected to 2 dimensional tensor
keys = wrd_embeddings@W_key
values = wrd_embeddings@W_value
query_2

tensor([0.4306, 1.4551], grad_fn=<SqueezeBackward4>)

In [91]:
keys

tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]], grad_fn=<MmBackward0>)

In [92]:
values

tensor([[0.1855, 0.8812],
        [0.3951, 1.0037],
        [0.3879, 0.9831],
        [0.2393, 0.5493],
        [0.1492, 0.3346],
        [0.3221, 0.7863]], grad_fn=<MmBackward0>)

In [99]:
# dot product now happens between queries and keys 
attn_scores_2 = query_2@keys.T
attn_scores_2[1] #score 2_2

tensor(1.8524, grad_fn=<SelectBackward0>)

In [ ]:
attn_scores_2
d_k = keys.shape[1]
torch.softmax(attn_scores_2,dim=0)
attn_weights_2 = torch.softmax(attn_scores_2/d_k**0.5,dim = 0) #new concept of division by square root of d_k as first mentioned in Attention Is All You Need paper
# d_k is a normalization factor that helps keep scores in a reasonable range
context_vector_2 = attn_weights_2@values
context_vector_2
# q,k,v architecture happens to give good performance empirically, research has shown pretty good results even after reducing the number of matrix multiplications


tensor([0.3061, 0.8210], grad_fn=<SqueezeBackward4>)

Formalizing the above

In [108]:
torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in,d_out)) # wrapper that makes weights trainable
W_key = torch.nn.Parameter(torch.rand(d_in,d_out))
W_value = torch.nn.Parameter(torch.rand(d_in,d_out))

d_in = wrd_embeddings.shape[1]
d_out = 2

query = wrd_embeddings@W_query #6,2
keys = wrd_embeddings@W_key
d_k = keys.shape[1]
values = wrd_embeddings@W_value

attn_weights = torch.softmax(query@keys.T/d_k**0.5, dim = 1) #6,6
context_vector = attn_weights@values
print(context_vector[1]) #quick check 



tensor([0.3061, 0.8210], grad_fn=<SelectBackward0>)


In [ ]:
# Implementing a self-attention class
from dataclasses import dataclass
import torch.nn as nn
# @dataclass
class SelfAttention_v1(nn.Module):
    def __init__(self,d_in,d_out,seed=123):
        super().__init__()
        torch.manual_seed(seed)
        self.W_query = torch.nn.Parameter(torch.rand(d_in,d_out))
        self.W_key = torch.nn.Parameter(torch.rand(d_in,d_out))
        self.W_value = torch.nn.Parameter(torch.rand(d_in,d_out))

    def forward(self,wrd_embeddings):
        query = wrd_embeddings@self.W_query
        keys = wrd_embeddings@self.W_key
        d_k = keys.shape[1]
        values = wrd_embeddings@self.W_value
        attn_scores = query@keys.T
        attn_weights = torch.softmax(attn_scores/d_k**0.5, dim = 1)
        context_vector = attn_weights@values

        return context_vector


sa_v1 = SelfAttention_v1(d_in,d_out)
sa_v1(wrd_embeddings)

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

In [ ]:
# %%writefile SelfAttention_v2.py
import torch.nn as nn

class SelfAttention_v2(nn.Module):
    def __init__(self,d_in,d_out,seed=123,qkv_bias=False):
        super().__init__()
        torch.manual_seed(seed)
        self.W_query = torch.nn.Linear(d_in,d_out,bias=qkv_bias) #slightly better weight initialization under the hood
        self.W_key = torch.nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value = torch.nn.Linear(d_in,d_out,bias=qkv_bias)

    def forward(self,wrd_embeddings):
        queries = self.W_query(wrd_embeddings)
        keys =  self.W_key(wrd_embeddings)
        d_k = keys.shape[1]
        values = self.W_value(wrd_embeddings)
        attn_scores = queries@keys.T
        attn_weights = torch.softmax(attn_scores/d_k**0.5, dim = 1)
        context_vector = attn_weights@values

        return context_vector


sa_v2 = SelfAttention_v2(d_in,d_out,789)
sa_v2(wrd_embeddings)

Writing SelfAttention_v2.py


### Hiding future words with causal attention

In [8]:
# Applying a causal attention mask
#  At each stage the LLM is supposed to only predict the next token, hence the future tokens need to be masked.
import torch.nn as nn

class SelfAttention_v3(nn.Module):
    def __init__(self,d_in,d_out,seed=123,qkv_bias=False):
        super().__init__()
        torch.manual_seed(seed)
        self.W_query = torch.nn.Linear(d_in,d_out,bias=qkv_bias) #slightly better weight initialization under the hood
        self.W_key = torch.nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value = torch.nn.Linear(d_in,d_out,bias=qkv_bias)

    def forward(self,wrd_embeddings):
        queries = self.W_query(wrd_embeddings)
        keys =  self.W_key(wrd_embeddings)
        d_k = keys.shape[1]
        values = self.W_value(wrd_embeddings)
        attn_scores = queries@keys.T
        mask = torch.triu(torch.ones(attn_scores.shape[0],attn_scores.shape[1]),diagonal=1)
        masked = attn_scores.masked_fill(mask.bool(),-torch.inf)
        attn_weights = torch.softmax(masked/d_k**0.5, dim = 1)
        context_vector = attn_weights@values
        return context_vector


sa_v2 = SelfAttention_v3(d_in,d_out,789)
sa_v2(wrd_embeddings)


tensor([[-0.0872,  0.0286],
        [-0.0991,  0.0501],
        [-0.0999,  0.0633],
        [-0.0983,  0.0489],
        [-0.0514,  0.1098],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)

## Masking additional attention weights with dropout

In [10]:
torch.manual_seed(123)

layer = torch.nn.Dropout(0.5)

In [12]:
example = torch.ones(6,6)

print(example)
print(layer(example))

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])
tensor([[0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.],
        [2., 0., 0., 0., 0., 0.],
        [0., 0., 2., 0., 2., 2.],
        [0., 2., 2., 2., 2., 2.]])


## Generate Self Attention class with the two new masks,the causal mask and drop mask

In [21]:
batch = torch.stack((wrd_embeddings,wrd_embeddings),dim=0)
batch.shape

torch.Size([2, 6, 3])

In [ ]:
# %%writefile CausalAttention.py
# Applying a causal attention and a dropout mask
#  At each stage the LLM is supposed to only predict the next token, hence the future tokens need to be masked.
import torch.nn as nn

class CausalAttention(nn.Module):
    def __init__(self,d_in,d_out,context_length,dropout,seed=123,qkv_bias=False):
        super().__init__()
        torch.manual_seed(seed)
        self.W_query = torch.nn.Linear(d_in,d_out,bias=qkv_bias) #slightly better weight initialization under the hood
        self.W_key = torch.nn.Linear(d_in,d_out,bias=qkv_bias)
        self.W_value = torch.nn.Linear(d_in,d_out,bias=qkv_bias)
        self.dropout = torch.nn.Dropout(dropout)
        self.register_buffer("mask",torch.triu(torch.ones(context_length,context_length),diagonal=1)) #register_buffer helps register tensors to move over to the GPU

    def forward(self,x):
        b,num_tokens,d_in = x.shape
        queries = self.W_query(x)
        keys =  self.W_key(x)
        d_k = keys.shape[1]
        values = self.W_value(x)
        attn_scores = queries@keys.T
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens,:num_tokens],-torch.inf) # in pytorch the _ suffix for function is inplace convention 
        attn_weights = torch.softmax(attn_scores/d_k**0.5, dim = 1)
        attn_weights= self.dropout(attn_weights)
        context_vector = attn_weights@values
        return context_vector

dropout = 0.0
context_length = batch.shape[1]
ca_v1 = CausalAttention(d_in,d_out,context_length,dropout,789)
ca_v1(wrd_embeddings)


Writing CausalAttention.py


## Stacking multiple single head attention layers